In [1]:
import pandas as pd
from scipy.stats import chi2_contingency

df_fase2 = pd.read_parquet("sharechat_fase2_etiquetado_v2.parquet")

# Tabla de contingencia solo EN vs ES
tabla = pd.crosstab(df_fase2["lang"], df_fase2["etiqueta"])
tabla_en_es = tabla.loc[["en", "es"]]

print("Tabla de contingencia EN vs ES:")
print(tabla_en_es)

print("\nPorcentajes:")
print(tabla_en_es.div(tabla_en_es.sum(axis=1), axis=0).round(3) * 100)

# Chi-cuadrada
chi2, p, dof, expected = chi2_contingency(tabla_en_es)

print(f"\nChi-cuadrada: {chi2:.4f}")
print(f"p-value:      {p:.4f}")
print(f"Grados de libertad: {dof}")
print(f"\nConclusión:")
if p < 0.05:
    print("Diferencia estadisticamente significativa (p < 0.05)")
    print("=> Hay evidencia de sesgo de alineacion entre EN y ES")
else:
    print("No hay diferencia estadisticamente significativa (p >= 0.05)")
    print("=> No hay evidencia suficiente de sesgo")

Tabla de contingencia EN vs ES:
etiqueta  CUMPLIMIENTO  CUMPLIMIENTO_CON_DISCLAIMER  RECHAZO
lang                                                        
en                 143                           75       12
es                  64                           41        6

Porcentajes:
etiqueta  CUMPLIMIENTO  CUMPLIMIENTO_CON_DISCLAIMER  RECHAZO
lang                                                        
en                62.2                         32.6      5.2
es                57.7                         36.9      5.4

Chi-cuadrada: 0.6689
p-value:      0.7157
Grados de libertad: 2

Conclusión:
No hay diferencia estadisticamente significativa (p >= 0.05)
=> No hay evidencia suficiente de sesgo


In [2]:
# Chi-cuadrada por plataforma
tabla_plataforma = pd.crosstab(df_fase2["platform"], df_fase2["etiqueta"])
print(tabla_plataforma)
print("\nPorcentajes:")
print(tabla_plataforma.div(tabla_plataforma.sum(axis=1), axis=0).round(3) * 100)

chi2, p, dof, _ = chi2_contingency(tabla_plataforma)
print(f"\nChi-cuadrada por plataforma: {chi2:.4f}")
print(f"p-value: {p:.4f}")

etiqueta    CUMPLIMIENTO  CUMPLIMIENTO_CON_DISCLAIMER  RECHAZO
platform                                                      
chatgpt               63                           26        2
claude                55                           21        8
gemini                33                           35        4
grok                  52                           25        6
perplexity            15                           13        0

Porcentajes:
etiqueta    CUMPLIMIENTO  CUMPLIMIENTO_CON_DISCLAIMER  RECHAZO
platform                                                      
chatgpt             69.2                         28.6      2.2
claude              65.5                         25.0      9.5
gemini              45.8                         48.6      5.6
grok                62.7                         30.1      7.2
perplexity          53.6                         46.4      0.0

Chi-cuadrada por plataforma: 19.5193
p-value: 0.0123


In [3]:
# Distribución completa: idioma X plataforma
tabla_lang_platform = pd.crosstab(df_fase2["lang"], df_fase2["platform"])
print("Filas por idioma y plataforma:")
print(tabla_lang_platform)

print("\nTotal por idioma:")
print(df_fase2["lang"].value_counts())

print("\nTotal por plataforma:")
print(df_fase2["platform"].value_counts())

Filas por idioma y plataforma:
platform  chatgpt  claude  gemini  grok  perplexity
lang                                               
en             12      81      59    59          19
es             76       0      11    17           7
other           3       3       2     7           2

Total por idioma:
lang
en       230
es       111
other     17
Name: count, dtype: int64

Total por plataforma:
platform
chatgpt       91
claude        84
grok          83
gemini        72
perplexity    28
Name: count, dtype: int64


In [4]:
# ¿Cuántas filas en español necesitas por plataforma?
print("Situación actual EN vs ES por plataforma:")
print(tabla_lang_platform)

print("\nPara tener balance necesitarías:")
for platform in ["claude", "gemini", "grok", "perplexity"]:
    en_count = tabla_lang_platform.loc["en", platform] if "en" in tabla_lang_platform.index else 0
    es_count = tabla_lang_platform.loc["es", platform] if "es" in tabla_lang_platform.index else 0
    faltantes = en_count - es_count
    print(f"  {platform}: tiene {es_count} ES, necesita ~{en_count} ES, faltan ~{faltantes}")

Situación actual EN vs ES por plataforma:
platform  chatgpt  claude  gemini  grok  perplexity
lang                                               
en             12      81      59    59          19
es             76       0      11    17           7
other           3       3       2     7           2

Para tener balance necesitarías:
  claude: tiene 0 ES, necesita ~81 ES, faltan ~81
  gemini: tiene 11 ES, necesita ~59 ES, faltan ~48
  grok: tiene 17 ES, necesita ~59 ES, faltan ~42
  perplexity: tiene 7 ES, necesita ~19 ES, faltan ~12


In [5]:
from datasets import load_dataset

# Verificar cuánto español hay en Claude
dataset = load_dataset("tucnguyen/ShareChat", name="claude", split="train", streaming=True)

count_es = 0
count_total = 0
MAX_ITER = 50_000

for i, fila in enumerate(dataset):
    if i >= MAX_ITER:
        break
    count_total += 1
    if fila.get("detected_language_final") == "Spanish" and fila["role"] == "user":
        count_es += 1

print(f"Total revisado: {count_total}")
print(f"Filas en español (user): {count_es}")
print(f"Proporcion ES: {count_es/count_total*100:.2f}%")

Total revisado: 8364
Filas en español (user): 114
Proporcion ES: 1.36%
